# 02-GRPO · SFT 다음 GRPO 정련 (RLHF) — 텍스트 분류(intent)

**TL;DR** — SFT로 학습한 모델을 **base로 이어받아** GRPO(Group Relative Policy Optimization)로 추가 정련합니다. 정답 모방(SFT) 위에, prompt당 여러 응답을 생성해 **reward 함수**로 좋은 응답을 강화합니다.

**Why** — 정석 RLHF 파이프라인은 **SFT → 정책최적화(GRPO/PPO)** 순서입니다. SFT로 형식·기본 능력을 갖춘 뒤 GRPO로 태스크 지표(추출=JSON 정확도, 분류=라벨 일치)를 직접 끌어올립니다. 이 트랙은 reward를 프로그램적으로 채점할 수 있어 GRPO에 적합합니다.

**기존 Pain Point** — base에서 바로 GRPO를 돌리면 형식조차 안 잡혀 rollout이 불안정합니다. SFT를 먼저 하면 GRPO가 안정적으로 수렴합니다. (요약·자유서술은 reward가 애매해 이 kit은 추출·분류 트랙에만 GRPO를 제공합니다.)

> 🔴 실제 실행 시 AWS 자격증명·GPU·엔드포인트 과금이 발생합니다. 먼저 `DRY_RUN=1`로 파이프라인을 검증하세요.

In [ ]:
import os, sys
# 리포 루트를 path에 추가해 common/ 와 트랙 로컬 모듈을 import
REPO = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.insert(0, REPO)
sys.path.insert(0, os.getcwd())

## GRPO vs SFT — 무엇이 다른가
| | SFT (`02_train_sft_sagemaker`) | **GRPO (이 노트북)** |
|---|---|---|
| 학습 신호 | 정답 completion 모방 | reward 함수로 생성물 강화 |
| 필요한 것 | `{messages}` (정답 포함) | prompt + **reward 함수**(정답은 채점용) |
| 연산량 | 1x | **크다** — prompt당 `num_generations`개 생성(rollout) |
| 적합 | 대부분의 태스크 | reward가 명확한 태스크(추출·분류) |

이 트랙의 reward: **`classification`** — 예측 라벨이 정답 라벨과 일치하는지 채점합니다(정확 1.0 / 부분 0.3 / 오답 0.0).
reward 로직은 `scripts/train_grpo.py`의 `reward_classification` 함수를 참고하세요.
GRPO는 rollout 때문에 SFT보다 학습이 오래 걸립니다 — 실습은 `MAX_TRAIN_SAMPLES`/`num_generations`를 낮춰 시작하세요.

In [ ]:
import importlib, boto3
from common import config, dlc, aws_utils; importlib.reload(config)
from sagemaker.core.helper.session_helper import Session
from sagemaker.train.model_trainer import ModelTrainer
from sagemaker.core.training.configs import SourceCode, Compute, InputData, StoppingCondition
sess = Session(boto3.Session(region_name=config.AWS_REGION))
%store -r role
%store -r md_classification
%store -r model_data   # SFT(02_train_sft_sagemaker)가 저장한 산출물 = GRPO의 base
model_data = globals().get('md_classification') or globals().get('model_data')
if 'role' not in dir() or not role or ':role/' not in str(role):
    role = config.resolve_sagemaker_role(sess)
# 🔴 리전 가드: %store 값은 리전을 바꿔도 남으므로 옛 리전 버킷을 가리킬 수 있습니다.
#    학습 잡도 같은 리전 S3만 읽으므로 여기서 현재 리전 최신 산출물로 맞춥니다.
from common import aws_utils
model_data = aws_utils.ensure_model_data_in_region(
    locals().get('model_data'), config.AWS_REGION, job_prefix='gemma-classification-train')
print('role      :', role)
print('SFT model :', model_data)   # 이 모델을 base로 GRPO를 이어서 학습합니다

## 1. GRPO용 prompt 준비 — 🔴 **SFT 데이터를 그대로 쓰면 안 됩니다**
SFT와 RL은 필요한 데이터가 다릅니다. GRPO는 `prompt`만 받아 스스로 생성하고 정답은 **reward 계산에만** 씁니다(`train_grpo.py`의 `_to_grpo`가 `{messages}`를 `{prompt, reference}`로 분해 — reference는 모델에 보여주지 않습니다).

**같은 데이터를 쓰면 학습이 아예 안 됩니다.** GRPO는 prompt당 rollout을 여러 개 생성해 **그룹 안에서 상대 비교**로 학습하는데, SFT가 이미 잘 맞히는 prompt는 rollout이 전부 만점이 되어 편차가 사라집니다 → **advantage ≈ 0 → gradient가 흐르지 않습니다.** GPU 시간만 쓰고 배우는 게 없습니다.
🔴 그래서 **같은 분포에서 슬라이스만 나눠도 부족합니다** — 누출은 막지만 advantage 문제는 남습니다.

| `GRPO_PROMPT_SOURCE` | 무엇 | 비용·선행조건 | advantage |
|---|---|---|---|
| `synth` (기본) | Bedrock으로 **prompt만** 생성 + 난이도 제약 | Bedrock 과금(소액) | ✅ 확보 |
| `failures` | `04_evaluate`에서 **틀린 건만** | 03·04 선행 필요 | **가장 강함** |
| `holdout` | SFT가 쓰지 않은 구간 | 무료·즉시 | ⚠️ 약함(같은 분포) |

🔴 **기본값이 `synth`인 이유**: `holdout`은 무료지만 같은 분포라 advantage가 잘 안 생깁니다. `synth`는 생성 프롬프트에 **난이도 제약**을 걸어 어려운 예시를 만듭니다 — 실측(추출 트랙): 제약 없이 합성하면 8건 전부 인자 0개(seed 분포가 인자 없는 함수 94%)였는데, 제약을 걸면 **인자 없음 0건 / 평균 인자 2.1개**가 되고 값을 간접 표현("the day after tomorrow")하는 입력이 나옵니다.
  (제약은 **생성 프롬프트에만** 넣습니다. critique에도 넣으면 seed와 다르다며 전부 기각합니다 — 실측 8/8 기각.)
`04_evaluate`를 이미 돌렸다면 **`failures`가 가장 효과적**입니다. 실전에서는 여기에 **실제 트래픽 로그**가 가장 좋은 소스입니다.
> 상세 근거: [`docs/03_finetuning.md` 「SFT에서 GRPO로 — 데이터를 갈아야 하는 이유」](../../docs/03_finetuning.md#sft에서-grpo로--데이터를-갈아야-하는-이유), 구현: `common/grpo_data.py`

In [ ]:
import os, importlib
# 🔴 common/* 를 고친 뒤 커널을 재시작하지 않으면 파이썬이 **옛 모듈을 캐시**해 계속 씁니다
#    (실측: 고친 aws_utils 가 반영되지 않아 Bedrock 응답 파싱이 계속 실패 → 합성 0건).
#    reload로 이 함정을 막습니다. 그래도 이상하면 Kernel → Restart 후 처음부터 실행하세요.
from common import aws_utils, grpo_data as gd
from common.synth import bedrock_synth as _bs
for _m in (aws_utils, _bs, gd):
    importlib.reload(_m)
import importlib, track_data as td; importlib.reload(td)

# prompt 소스: 'synth'(기본) | 'failures' | 'holdout'
GRPO_PROMPT_SOURCE = 'synth'
N_GRPO = 100          # GRPO는 prompt당 rollout N개라 느립니다 — 작게 시작하세요.

if GRPO_PROMPT_SOURCE == 'holdout':
    # SFT가 쓴 앞 NUM_SEED_SAMPLES건 '이후' 구간 → 누출 방지(단 같은 분포)
    rows = gd.from_holdout('data/train.jsonl', N_GRPO, sft_used=config.NUM_SEED_SAMPLES)
elif GRPO_PROMPT_SOURCE == 'synth':
    # ⚠️ SFT 합성과 같은 시드를 주면 분포가 또 겹칩니다 → SFT 미사용 시드 구간을 넘깁니다.
    pool = td.load_seed_examples(config.NUM_SEED_SAMPLES + N_GRPO, token=config.get_hf_token())
    fresh = pool[config.NUM_SEED_SAMPLES:]
    rows = gd.from_synth(task_instruction=td.TASK_INSTRUCTION,
                        seed_texts=td.seed_texts_for_synth(fresh),
                        n=N_GRPO, model_id=config.BEDROCK_CLAUDE_MODEL_ID,
                        region=config.BEDROCK_REGION, to_messages=td.to_messages,
                        kind='classification')   # 난이도 제약 적용
else:  # 'failures' — 04_evaluate 를 먼저 실행해 heldout/preds 가 커널에 있어야 합니다.
    assert 'preds' in dir() and 'heldout' in dir(), (
        "failures 소스는 04_evaluate 의 (heldout, preds)가 필요합니다.\n"
        '  → 04_evaluate 를 먼저 실행한 뒤 같은 커널에서 이 셀을 돌리거나, 그 결과를 저장해 불러오세요.')
    rows = gd.from_failures(heldout, preds, kind='classification',
                            to_messages=td.to_messages, max_n=N_GRPO)

gd.describe(rows, source=GRPO_PROMPT_SOURCE)
train_path = gd.write_grpo_jsonl(rows, 'data/grpo_train.jsonl')
bucket = config.S3_BUCKET or sess.default_bucket()
key = f'{config.S3_PREFIX}/classification/grpo/train.jsonl'
train_s3 = aws_utils.upload_if_changed(train_path, bucket, key, config.AWS_REGION)
print('train_s3:', train_s3)

## 2. GRPO ModelTrainer 구성
`entry_script`가 `train_grpo.py`이고, `reward_kind`로 이 트랙의 채점 방식을 지정합니다. `num_generations`는 prompt당 생성 수(그룹 크기)로, 클수록 학습 신호가 좋아지지만 연산량이 늘어납니다.

In [ ]:
# 학습 건수는 §1의 N_GRPO 가 이미 결정합니다(prompt 소스에서 그만큼만 뽑음).
#    여기서 더 줄이려면 MAX_TRAIN_SAMPLES 를 정수로 두세요(None이면 §1이 만든 전량).
MAX_TRAIN_SAMPLES = None
MAX_RUNTIME_HOURS = 6   # GRPO는 rollout 때문에 SFT보다 오래 걸립니다
hyperparameters = {
    # model_id는 멀티모달 감지 fallback용. 실제 base는 아래 'model' 채널(SFT 산출물)에서 로드.
    'model_id': config.DEFAULT_MODEL_ID,
    'reward_kind': 'classification',
    'epochs': 1, 'per_device_train_batch_size': 1, 'gradient_accumulation_steps': 8,
    'learning_rate': 1e-5,
    'num_generations': 8, 'max_completion_length': 256,
    'max_seq_length': 512,
    'lora_r': 16, 'lora_alpha': 16, 'lora_dropout': 0.05,
    'use_qlora': True, 'merge_adapter': True,
}
if MAX_TRAIN_SAMPLES:
    hyperparameters['max_train_samples'] = MAX_TRAIN_SAMPLES
environment = {'HF_TOKEN': config.get_hf_token()} if config.get_hf_token() else {}
image_uri = dlc.resolve_training_image(config.AWS_REGION)
assert image_uri, 'DLC 이미지 해석 실패 — DLC_IMAGE_URI env로 지정: ' + dlc.AVAILABLE_IMAGES_URL
trainer = ModelTrainer(
    training_image=image_uri,
    source_code=SourceCode(source_dir='scripts', entry_script='train_grpo.py',
                           requirements='requirements.txt'),
    compute=Compute(instance_type=config.TRAIN_INSTANCE_TYPE, instance_count=1),
    hyperparameters=hyperparameters,
    environment=environment,
    role=role,
    sagemaker_session=sess,
    base_job_name='gemma-classification-grpo',
    stopping_condition=StoppingCondition(max_runtime_in_seconds=MAX_RUNTIME_HOURS * 3600),
)

## 3. 학습 시작 (비동기 제출) — SFT 산출물을 base로 마운트
🔴 **`model` 채널로 SFT 산출물(`model_data`)을 마운트**합니다. 컨테이너 안 `SM_CHANNEL_MODEL`(=`/opt/ml/input/data/model`)에 풀리고, `train_grpo.py`가 `--base_model_dir`로 이를 base로 받아 GRPO를 이어서 학습합니다(정석 SFT→GRPO). 상태 확인·세션 재접속은 SFT 노트북 §4~§5와 동일합니다(`TrainingJob.get(name)`로 재조회).

In [ ]:
trainer.train(input_data_config=[
        InputData(channel_name='train', data_source=train_s3),
        InputData(channel_name='model', data_source=model_data),  # SFT 산출물 = GRPO base
    ], wait=False, logs=False)
from IPython.display import display
job = trainer._latest_training_job
print('GRPO training job:', job.training_job_name)
display(aws_utils.cw_links(config.AWS_REGION, training_job=job.training_job_name))

In [ ]:
aws_utils.training_job_status(job.training_job_name, config.AWS_REGION)

## 4. 완료 대기 → 모델 아티팩트
완료되면 SFT와 동일하게 `model_data`(S3)가 나오고, 멀티모달 base면 `train_grpo.py`가 텍스트 전용으로 re-export해 저장합니다. 이후 **03_deploy_endpoint**로 배포합니다(SFT와 동일).

In [ ]:
import time
while True:
    job.refresh()
    st = job.training_job_status
    print('status:', st)
    if st in ('Completed', 'Failed', 'Stopped'):
        break
    time.sleep(30)
assert st == 'Completed', f'GRPO 잡이 {st} 상태입니다. CloudWatch 로그를 확인하세요.'
grpo_model_data = job.model_artifacts.s3_model_artifacts
print('GRPO complete. Model artifact:', grpo_model_data)
# 이후 02b/03은 model_data를 서빙합니다. GRPO 결과를 배포하려면 model_data를 GRPO 산출물로 갱신:
model_data = grpo_model_data
md_classification = grpo_model_data   # 이 트랙 전용 키도 갱신
%store model_data
%store md_classification
%store grpo_model_data   # SFT와 비교하려면 각각의 URI를 따로 보관
print('model_data -> GRPO 산출물로 설정됨 (02b/03이 이걸 서빙)')

✅ GRPO 학습이 끝났습니다. `model_data`가 GRPO 산출물을 가리키도록 갱신됐으니 **03_deploy_endpoint.ipynb**(또는 02b 로컬 검증)로 그대로 배포하면 됩니다.
> SFT vs GRPO 성능 비교: `grpo_model_data`와 SFT의 URI를 각각 배포/평가(04_evaluate)해 지표를 비교하세요. SFT만 배포하려면 02_train_sft의 `model_data`를 다시 `%store` 하면 됩니다.